Exercise task 5: Feature importance

Part A:

Just copying the lectures notebook code for tokenizer:

In [1]:
import datasets
import evaluate
import transformers
import torch
import numpy as np
from pprint import pprint
from sklearn import metrics

In [2]:
dset=datasets.load_dataset("imdb")
pprint(dset)
dset=dset.shuffle()
del dset["unsupervised"]
pprint(dset['train'][0]['text'])
print(dset['train'][0]['label'])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
('This is a beautiful, rich, and very well-executed film with a rich and '
 'meaningful story. Basically, it tells how an old master story teller needs '
 'to find a (male) heir to carry on his craft, but ends up not getting what he '
 'expected in his very male-dominated world. The characters must then deal '
 'with their situation and the old master must grapple with the conflict '
 "between his desire for a companion and heir and his and society's "
 'traditional notions.<br /><br />The story is fun, emotional, and complex. '
 'The exploration of the characters, their lives, and emotions, is rich and '
 'compelling the character development is strong while the characters are '
 'complex and not one

Tokenizer:

In [3]:
base_tokenizer = transformers.AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
tokenizer = base_tokenizer.train_new_from_iterator(
    dset["train"]["text"],
    vocab_size=15000
)

In [4]:
def encode(examples):
    return tokenizer(examples['text'],
                     #truncation=True,
                     #max_length=256
                     )

dset_tokenized = dset.map(encode,batched=True,num_proc=4)

for key,val in dset_tokenized["train"][0].items():
    print(key,":",val)

Map (num_proc=4):   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (657 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1105 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (826 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (933 > 512). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/25000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (534 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (536 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (864 > 512). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (1386 > 512). Running this sequence through the model will result in indexing errors


text : This is a beautiful, rich, and very well-executed film with a rich and meaningful story. Basically, it tells how an old master story teller needs to find a (male) heir to carry on his craft, but ends up not getting what he expected in his very male-dominated world. The characters must then deal with their situation and the old master must grapple with the conflict between his desire for a companion and heir and his and society's traditional notions.<br /><br />The story is fun, emotional, and complex. The exploration of the characters, their lives, and emotions, is rich and compelling the character development is strong while the characters are complex and not one dimensional at all. The film expertly conveys the old man's emotions and his desire to find an heir, and compellingly shows how he and the kid handle the situation. There is also humour, sometimes quite subtle, at appropriate points. The film also examines the good and bad of traditional Chinese culture, creating furth

In [5]:
collator=transformers.DataCollatorWithPadding(tokenizer)
small_data=[tokenizer("Hi there"), tokenizer("A little longer text!")]
print("small_data:\n")
pprint(small_data)
print("\n\ncollated:\n")
small_batch=collator(small_data)
pprint(small_batch)


small_data:

[{'attention_mask': [1, 1, 1, 1],
  'input_ids': [2, 6004, 310, 3],
  'token_type_ids': [0, 0, 0, 0]},
 {'attention_mask': [1, 1, 1, 1, 1, 1, 1],
  'input_ids': [2, 43, 566, 3056, 4274, 5, 3],
  'token_type_ids': [0, 0, 0, 0, 0, 0, 0]}]


collated:

{'attention_mask': tensor([[1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1]]),
 'input_ids': tensor([[   2, 6004,  310,    3,    0,    0,    0],
        [   2,   43,  566, 3056, 4274,    5,    3]]),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0]])}


In [ ]:
# A Transformers library model wants a config,
# I can simply inherit from the base
# class for pretrained configs
# We don't really need to do anything very special
class MLPConfig(transformers.PretrainedConfig):
    pass

# This is the model
class MLP(transformers.PreTrainedModel):

    config_class=MLPConfig

    # In the initialization method, one instantiates the layers
    # these will be, for the most part the trained parameters of the model
    def __init__(self,config):
        super().__init__(config)
        self.all_tied_weights_keys = {} #Annoying bug in Transformers, must have this here or else we crash on saved model load
        #### HERE WE CREATE THE MODEL'S LAYERS:
        self.vocab_size=config.vocab_size #embedding matrix row count
        # Build and initialize embedding of vocab size x hidden size
        assert tokenizer.vocab["[PAD]"]==0 #let's make sure our assumption of pad==0 holds!

        self.embedding=torch.nn.Embedding(num_embeddings=self.vocab_size,embedding_dim=config.hidden_size,padding_idx=0)
        # Initialize the embeddings to random values
        # Note! This function is relatively clever and keeps the embedding for 0, the padding, pure zeros
        torch.nn.init.uniform_(self.embedding.weight.data,-0.001,0.001) #initialize the embeddings with small random values

        # This takes care of the lower half of the network, now the upper half
        # Output layer: hidden size x output size
        self.output=torch.nn.Linear(in_features=config.hidden_size,out_features=config.nlabels)
        # Now we have the parameters of the model
        self.loss=torch.nn.CrossEntropyLoss() #This loss is meant for classification, so let's use it


    # The computation of the model is put into the forward() function
    # it receives a batch of data and optionally the correct `labels`
    #
    # If given `labels` we should return (loss,output)
    # if not, then we should return (output,)
    # that way the model can be used both for training and for inference
    def forward(self,input_ids,labels=None,**kwargs):
        #1) sum up the embeddings of the items
        embedded=self.embedding(input_ids) #(batch,ids)->(batch,ids,embedding_dim)
        # Since the Embedding keeps the first row of the matrix pure zeros, we don't need to worry about the padding 0 index
        # so next we sum the embeddings across the word dimension
        # (batch,ids,embedding_dim) -> (batch,embedding_dim)
        embedded_summed=torch.sum(embedded,dim=1)

        #2) apply non-linearity
        # (batch,embedding_dim) -> (batch,embedding_dim)
        projected=torch.tanh(embedded_summed) #Note how non-linearity is applied here and not when configuring the layer in __init__()

        #3) and now apply the upper, output layer of the network
        # (batch,embedding_dim) -> (batch, num_of_classes i.e. 2 in our case)
        logits=self.output(projected)

        # ...and that's all there is to it!

        #print("input_ids.shape",input_ids.shape)
        #print("embedded.shape",embedded.shape)
        #print("embedded_summed.shape",embedded_summed.shape)
        #print("projected.shape",projected.shape)
        #print("logits.shape",logits.shape)

        # If we have labels, we ought to calculate the loss
        if labels is not None:
            # You run the loss as loss(model_output,correct_labels)
            return (self.loss(logits,labels),logits) #2-tuple, i.e. pair of values returned
        else:
            # No labels, so just return the logits
            return (logits,) #this weird syntax means 1-tuple



In [7]:
# Configure the model:
#   these parameters are used in the model's __init__()
mlp_config=MLPConfig(vocab_size=tokenizer.vocab_size,hidden_size=20,nlabels=2)
print("mlp config:", mlp_config)

# And now we can instantiate it
mlp=MLP(mlp_config)
print("mlp",mlp)
#we can make a little test with the small test batch we made earlier
#since it has no true labels, it should return a 1-tuple, which it will
out=mlp(input_ids=small_batch["input_ids"])
print("Output on one batch:",out)

mlp config: MLPConfig {
  "hidden_size": 20,
  "nlabels": 2,
  "transformers_version": "5.3.0",
  "vocab_size": 15000
}

mlp MLP(
  (embedding): Embedding(15000, 20, padding_idx=0)
  (output): Linear(in_features=20, out_features=2, bias=True)
  (loss): CrossEntropyLoss()
)
Output on one batch: (tensor([[0.1557, 0.1689],
        [0.1542, 0.1699]], grad_fn=<AddmmBackward0>),)


In [8]:
trainer_args = transformers.TrainingArguments(
    "mlp_checkpoints", #save checkpoints here
    eval_strategy="steps", #...and not epochs (step is "one batch", epoch is "one full pass through the whole data")
    logging_strategy="steps",
    eval_steps=500, #eval every 500 steps
    logging_steps=500,
    learning_rate=5e-5, #learning rate of the gradient descent
    max_steps=10000,
    load_best_model_at_end=True, #when done, load the best model you have (which is not necessarily the one after the last step)
    per_device_train_batch_size=16 #batch size
)

pprint(trainer_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=500,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False

In [9]:
accuracy = evaluate.load("accuracy")

def compute_accuracy(outputs_and_labels):
    outputs, labels = outputs_and_labels
    predictions = np.argmax(outputs, axis=-1) #pick the index of the "winning" label among the outputs, i.e. argmax
    return accuracy.compute(predictions=predictions, references=labels)

In [10]:
# Make a new model, this will also initialize it
mlp = MLP(mlp_config)


# Argument gives the number of evaluation tries of patience before early stopping
# i.e. training is stopped when the evaluation loss fails to improve
# certain number of times the model is evaluated, in this case 5 consecutive times
early_stopping = transformers.EarlyStoppingCallback(5)

trainer = transformers.Trainer(
    model=mlp,
    args=trainer_args,
    train_dataset=dset_tokenized["train"],
    eval_dataset=dset_tokenized["test"].select(range(1000)), #make a smaller subset to evaluate on
    compute_metrics=compute_accuracy,
    data_collator=collator,
    callbacks=[early_stopping]
)

# FINALLY!
# a bit slow on CPU but doable in about 4min
# about 1.5x faster on GPU (this neural net is too simple to really make the CPU/GPU difference stand out)
trainer.train()

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
500,0.611880,0.546094,0.825000
1000,0.493100,0.460253,0.842000
1500,0.427466,0.406124,0.860000
2000,0.358955,0.371001,0.864000
2500,0.332833,0.343900,0.869000
3000,0.311292,0.326118,0.876000
3500,0.280026,0.318995,0.872000
4000,0.265495,0.302701,0.889000
4500,0.260786,0.294758,0.886000
5000,0.240502,0.289559,0.888000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10000, training_loss=0.2854598213195801, metrics={'train_runtime': 56.7661, 'train_samples_per_second': 2818.582, 'train_steps_per_second': 176.161, 'total_flos': 36090667872.0, 'train_loss': 0.2854598213195801, 'epoch': 6.397952655150352})

In [ ]:
mlp.save_pretrained("mlp-imdb2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
mlp2=MLP.from_pretrained("mlp-imdb2")

Loading weights:   0%|          | 0/3 [00:00<?, ?it/s]

In [13]:
# The same trainer with slightly different arguments
# can be used to run prediction and get scores

eval_args = transformers.TrainingArguments(
  do_train=False,
  do_eval=False
)

trainer = transformers.Trainer(
    model=mlp2,
    args=eval_args,
    compute_metrics=compute_accuracy,
    data_collator=collator
)



In [14]:
eval_results = trainer.predict(dset_tokenized["test"])
print(eval_results)
print('Accuracy:', eval_results.metrics['test_accuracy'])

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


PredictionOutput(predictions=array([[-1.6606075 ,  2.20327   ],
       [ 0.39727718, -0.30637985],
       [ 2.4637659 , -2.9151735 ],
       ...,
       [-0.48612583,  0.8777754 ],
       [ 0.66763645, -0.72330385],
       [ 0.34199473, -0.3511787 ]], shape=(25000, 2), dtype=float32), label_ids=array([1, 0, 0, ..., 1, 0, 0], shape=(25000,)), metrics={'test_loss': 0.2829650044441223, 'test_model_preparation_time': 0.0001, 'test_accuracy': 0.88828, 'test_runtime': 6.4284, 'test_samples_per_second': 3889.0, 'test_steps_per_second': 486.125})
Accuracy: 0.88828


Now my own part:

In [15]:
weights=mlp.embedding.weight.detach().cpu().numpy()

Running the code to find nearest neighbor for the token 'study'. This could be interpreted as a noun or as a verb.

In [16]:
qry_idx=tokenizer.vocab["study"] #embedding of "study"
idx2word={v:k for k,v in tokenizer.vocab.items()}

#calculate the distance of the  embedding to all other embeddings
distance_to_qry=metrics.pairwise.euclidean_distances(weights[qry_idx:qry_idx+1,:],weights)
nearest_neighbors=np.argsort(distance_to_qry) #indices of nearest words
print("*Top of the list:*")
for nearest in nearest_neighbors[0,:20]:
    print(idx2word[nearest])
print("\n(...)\n")
print("*Bottom of the list:*")
for nearest in nearest_neighbors[0,-20:]:
    print(idx2word[nearest])


*Top of the list:*
study
##uction
agenda
##uable
tone
[PAD]
cav
##olds
##sel
enterpr
##oes
fore
contributes
##amped
reception
reid
dollars
##outs
##itel
superman

(...)

*Bottom of the list:*
badly
fails
7
pointless
great
wonderful
perfect
worse
poor
avoid
horrible
dull
poorly
terrible
bad
excellent
boring
awful
waste
worst


In [ ]:
qry_idx=tokenizer.vocab["dog"] #embedding of "dog"
idx2word={v:k for k,v in tokenizer.vocab.items()}

#calculate the distance of the  embedding to all other embeddings
distance_to_qry=metrics.pairwise.euclidean_distances(weights[qry_idx:qry_idx+1,:],weights)
nearest_neighbors=np.argsort(distance_to_qry) #indices of nearest words
print("*Top of the list:*")
for nearest in nearest_neighbors[0,:20]:
    print(idx2word[nearest])
print("\n(...)\n")
print("*Bottom of the list:*")
for nearest in nearest_neighbors[0,-20:]:
    print(idx2word[nearest])

*Top of the list:*
dog
laced
rocket
deny
petty
esquire
##cer
marion
contributed
muni
pornographic
bosses
##quest
guest
their
tour
immedi
throne
colleague
biting

(...)

*Bottom of the list:*
wonderful
perfect
disappointment
ridiculous
badly
fails
pointless
worse
poor
avoid
horrible
dull
excellent
poorly
terrible
bad
boring
awful
waste
worst


In [18]:
qry_idx=tokenizer.vocab["and"] #embedding of "and"
idx2word={v:k for k,v in tokenizer.vocab.items()}

#calculate the distance of the  embedding to all other embeddings
distance_to_qry=metrics.pairwise.euclidean_distances(weights[qry_idx:qry_idx+1,:],weights)
nearest_neighbors=np.argsort(distance_to_qry) #indices of nearest words
print("*Top of the list:*")
for nearest in nearest_neighbors[0,:20]:
    print(idx2word[nearest])
print("\n(...)\n")
print("*Bottom of the list:*")
for nearest in nearest_neighbors[0,-20:]:
    print(idx2word[nearest])

*Top of the list:*
and
overtones
parts
music
##acious
raunch
comedy
ireland
kicks
alot
flair
flow
hope
suspense
longest
##iture
saw
feel
hot
bay

(...)

*Bottom of the list:*
laughable
lame
disappointment
ridiculous
badly
fails
pointless
excellent
worse
poor
avoid
horrible
dull
poorly
terrible
bad
boring
awful
waste
worst


In these token cases I did expect the closest neighbours to be tokens that have some type of connection context or token vise to the original token, though I didn't expect the bottom of the list to be pretty much identical in every case. I was more expecting the bottom of the list to be more like a opposite word to the given token, like cat and dog or something like that.

Part B:

The changes I made to the above code is to remove the tahn function to make the code linear, and changed the hidden layer size to 1.

In [19]:
# A Transformers library model wants a config,
# I can simply inherit from the base
# class for pretrained configs
# We don't really need to do anything very special
class MLPConfig(transformers.PretrainedConfig):
    pass

# This is the model
class MLP(transformers.PreTrainedModel):

    config_class=MLPConfig

    # In the initialization method, one instantiates the layers
    # these will be, for the most part the trained parameters of the model
    def __init__(self,config):
        super().__init__(config)
        self.all_tied_weights_keys = {} #Annoying bug in Transformers, must have this here or else we crash on saved model load
        #### HERE WE CREATE THE MODEL'S LAYERS:
        self.vocab_size=config.vocab_size #embedding matrix row count
        # Build and initialize embedding of vocab size x hidden size
        assert tokenizer.vocab["[PAD]"]==0 #let's make sure our assumption of pad==0 holds!

        self.embedding=torch.nn.Embedding(num_embeddings=self.vocab_size,embedding_dim=config.hidden_size,padding_idx=0)
        # Initialize the embeddings to random values
        # Note! This function is relatively clever and keeps the embedding for 0, the padding, pure zeros
        torch.nn.init.uniform_(self.embedding.weight.data,-0.001,0.001) #initialize the embeddings with small random values

        # This takes care of the lower half of the network, now the upper half
        # Output layer: hidden size x output size
        self.output=torch.nn.Linear(in_features=config.hidden_size,out_features=config.nlabels)
        # Now we have the parameters of the model
        self.loss=torch.nn.CrossEntropyLoss() #This loss is meant for classification, so let's use it


    # The computation of the model is put into the forward() function
    # it receives a batch of data and optionally the correct `labels`
    #
    # If given `labels` we should return (loss,output)
    # if not, then we should return (output,)
    # that way the model can be used both for training and for inference
    def forward(self,input_ids,labels=None,**kwargs):
        #1) sum up the embeddings of the items
        embedded=self.embedding(input_ids) #(batch,ids)->(batch,ids,embedding_dim)
        # Since the Embedding keeps the first row of the matrix pure zeros, we don't need to worry about the padding 0 index
        # so next we sum the embeddings across the word dimension
        # (batch,ids,embedding_dim) -> (batch,embedding_dim)
        embedded_summed=torch.sum(embedded,dim=1)

        #2) apply non-linearity
        # (batch,embedding_dim) -> (batch,embedding_dim)

        ### Commented the tanh function out ###
        # projected=torch.tanh(embedded_summed) #Note how non-linearity is applied here and not when configuring the layer in __init__()

        #3) and now apply the upper, output layer of the network
        # (batch,embedding_dim) -> (batch, num_of_classes i.e. 2 in our case)
        logits=self.output(embedded_summed)

        # ...and that's all there is to it!

        #print("input_ids.shape",input_ids.shape)
        #print("embedded.shape",embedded.shape)
        #print("embedded_summed.shape",embedded_summed.shape)
        #print("projected.shape",projected.shape)
        #print("logits.shape",logits.shape)

        # If we have labels, we ought to calculate the loss
        if labels is not None:
            # You run the loss as loss(model_output,correct_labels)
            return (self.loss(logits,labels),logits) #2-tuple, i.e. pair of values returned
        else:
            # No labels, so just return the logits
            return (logits,) #this weird syntax means 1-tuple



In [20]:
# Configure the model:
#   these parameters are used in the model's __init__()
mlp_config=MLPConfig(vocab_size=tokenizer.vocab_size,hidden_size=1,nlabels=2)
print("mlp config:", mlp_config)

# And now we can instantiate it
mlp=MLP(mlp_config)
print("mlp",mlp)
#we can make a little test with the small test batch we made earlier
#since it has no true labels, it should return a 1-tuple, which it will
out=mlp(input_ids=small_batch["input_ids"])
print("Output on one batch:",out)

mlp config: MLPConfig {
  "hidden_size": 1,
  "nlabels": 2,
  "transformers_version": "5.3.0",
  "vocab_size": 15000
}

mlp MLP(
  (embedding): Embedding(15000, 1, padding_idx=0)
  (output): Linear(in_features=1, out_features=2, bias=True)
  (loss): CrossEntropyLoss()
)
Output on one batch: (tensor([[-0.4093,  0.2598],
        [-0.4112,  0.2579]], grad_fn=<AddmmBackward0>),)


In [21]:
# Make a new model, this will also initialize it
mlp = MLP(mlp_config)


# Argument gives the number of evaluation tries of patience before early stopping
# i.e. training is stopped when the evaluation loss fails to improve
# certain number of times the model is evaluated, in this case 5 consecutive times
early_stopping = transformers.EarlyStoppingCallback(5)

trainer = transformers.Trainer(
    model=mlp,
    args=trainer_args,
    train_dataset=dset_tokenized["train"],
    eval_dataset=dset_tokenized["test"].select(range(1000)), #make a smaller subset to evaluate on
    compute_metrics=compute_accuracy,
    data_collator=collator,
    callbacks=[early_stopping]
)

# FINALLY!
# a bit slow on CPU but doable in about 4min
# about 1.5x faster on GPU (this neural net is too simple to really make the CPU/GPU difference stand out)
trainer.train()

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
500,0.727822,0.701572,0.523000
1000,0.688067,0.674570,0.551000
1500,0.659676,0.650361,0.591000
2000,0.633314,0.625785,0.625000
2500,0.610687,0.604447,0.653000
3000,0.588775,0.584375,0.681000
3500,0.568436,0.567165,0.710000
4000,0.549139,0.550938,0.726000
4500,0.534510,0.537538,0.744000
5000,0.517365,0.525071,0.764000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10000, training_loss=0.5407430282592773, metrics={'train_runtime': 55.0936, 'train_samples_per_second': 2904.151, 'train_steps_per_second': 181.509, 'total_flos': 3437206464.0, 'train_loss': 0.5407430282592773, 'epoch': 6.397952655150352})

In [22]:
mlp.save_pretrained("mlp-imdb3")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
mlp3=MLP.from_pretrained("mlp-imdb3")

Loading weights:   0%|          | 0/3 [00:00<?, ?it/s]

In [24]:
eval_results = trainer.predict(dset_tokenized["test"])
print(eval_results)
print('Accuracy:', eval_results.metrics['test_accuracy'])

/home/lauris/Documents/school/2025-26/HLT/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


PredictionOutput(predictions=array([[-0.5378301 ,  0.3474722 ],
       [ 0.10129173, -0.7438999 ],
       [ 0.23290022, -0.9686361 ],
       ...,
       [-0.17850465, -0.2661163 ],
       [-0.10392279, -0.39347315],
       [-0.43150473,  0.16590965]], shape=(25000, 2), dtype=float32), label_ids=array([1, 0, 0, ..., 1, 0, 0], shape=(25000,)), metrics={'test_loss': 0.47739842534065247, 'test_accuracy': 0.80976, 'test_runtime': 6.1847, 'test_samples_per_second': 4042.216, 'test_steps_per_second': 505.277})
Accuracy: 0.80976


In [25]:
weights=mlp.embedding.weight.detach().cpu().numpy()
sorted_list=weights.flatten().argsort()

In [28]:
print("lowest weights (positives):")
for token in sorted_list[:20]:
    print(idx2word[token])
print("Highest weights (negatives):")
for token in sorted_list[-20:]:
    print(idx2word[token])

lowest weights (positives):
great
[CLS]
[SEP]
excellent
wonderful
best
perfect
loved
amazing
favorite
love
highly
enjoyed
beautiful
well
fantastic
brilliant
superb
today
very
Highest weights (negatives):
dull
supposed
avoid
ridiculous
pointless
crap
poorly
stupid
nothing
?
no
poor
horrible
boring
worse
terrible
awful
waste
bad
worst
